## DBSCAN und PCA-Reduktion

#### Daten unter data/df_piv_02.pkl

### Teil 1

In [ ]:
"""
DBSCAN-basierte Anomalie-Erkennung mit optionaler PCA-Reduktion.

Drei Betriebsarten für die Dimensionsreduktion:
  - "keine":          DBSCAN auf allen Originalmerkmalen, Visualisierung ebenso.
  - "visualisierung": DBSCAN auf allen Originalmerkmalen, Visualisierung auf
                      PCA-Hauptkomponenten (Empfehlung ab ~8 Merkmalen).
  - "erkennung":      DBSCAN auf PCA-Hauptkomponenten (nur bei sehr hoher
                      Dimensionalität oder starken Korrelationen sinnvoll).

Zahl der PCA-Komponenten wird entweder explizit gesetzt oder automatisch über
eine Varianz-Schwelle bestimmt (Default: 95 %).
"""

import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import DBSCAN
from sklearn.neighbors import NearestNeighbors


# ---------------------------------------------------------------------------
# Kern-Funktionen
# ---------------------------------------------------------------------------

def bestimme_eps(X: np.ndarray, min_samples: int) -> tuple[float, np.ndarray, int]:
    """Bestimmt eps über den k-Distance-Plot (Knick-Methode)."""
    nn = NearestNeighbors(n_neighbors=min_samples + 1).fit(X)
    distances, _ = nn.kneighbors(X)
    k_distances = np.sort(distances[:, -1])

    x_norm = np.linspace(0.0, 1.0, len(k_distances))
    y_norm = (k_distances - k_distances.min()) / (k_distances.max() - k_distances.min())
    knee_idx = int(np.argmax(np.abs(y_norm - x_norm)))
    return float(k_distances[knee_idx]), k_distances, knee_idx


def _bestimme_pca_komponenten(pca: PCA, komponenten: int | None,
                              varianz_schwelle: float) -> int:
    """Ermittelt die Zahl der PCA-Komponenten (explizit oder über Varianz)."""
    if komponenten is not None:
        return min(komponenten, pca.n_components_)
    cumvar = np.cumsum(pca.explained_variance_ratio_)
    return int(np.searchsorted(cumvar, varianz_schwelle) + 1)


def dbscan_anomalien(
    daten: np.ndarray,
    feature_namen: np.ndarray | list[str] | None = None,
    min_samples: int | None = None,
    eps: float | None = None,
    pca_modus: str = "keine",
    pca_komponenten: int | None = None,
    pca_varianz_schwelle: float = 0.95,
) -> dict:
    """
    DBSCAN-Anomalie-Erkennung mit optionaler PCA.

    Args:
        daten: (n_samples, n_features) Rohdaten.
        feature_namen: Namen der Merkmale (numpy-Array oder Liste), Länge = n_features.
                       Default: ["x1", "x2", ...]. Wird im Ergebnis-Dict abgelegt und
                       von den Plot-Funktionen automatisch übernommen.
        min_samples: Nachbarn im eps-Radius für Kernpunkte.
                     Default: max(4, 2 * n_features_der_Erkennung).
        eps: Nachbarschaftsradius. Falls None, automatisch bestimmt.
        pca_modus: "keine" | "visualisierung" | "erkennung".
        pca_komponenten: explizite Anzahl der Hauptkomponenten (überschreibt
                         pca_varianz_schwelle).
        pca_varianz_schwelle: minimale kumulative Varianz für die Auswahl der
                              Komponenten, wenn pca_komponenten=None.

    Returns:
        dict mit labels, anomalie-Maske, Parametern, k_distances, feature_namen,
        PCA-Objekt (falls verwendet), transformierten Daten, Scaler, etc.
    """
    if daten.ndim != 2:
        raise ValueError(f"Erwarte 2D-Array, bekomme {daten.ndim}D.")
    if pca_modus not in ("keine", "visualisierung", "erkennung"):
        raise ValueError(f"pca_modus muss 'keine', 'visualisierung' oder 'erkennung' sein.")

    n_samples, n_features = daten.shape

    # Feature-Namen normalisieren (akzeptiert np.array, list, tuple)
    if feature_namen is None:
        feature_namen_list = [f"x{i + 1}" for i in range(n_features)]
    else:
        feature_namen_list = [str(x) for x in feature_namen]
        if len(feature_namen_list) != n_features:
            raise ValueError(
                f"feature_namen hat Länge {len(feature_namen_list)}, "
                f"Daten haben aber {n_features} Merkmale."
            )

    # 1) Standardisierung
    scaler = StandardScaler()
    X_scaled = scaler.fit_transform(daten)

    # 2) PCA (falls angefordert)
    pca = None
    X_pca_full = None
    n_pca = None
    if pca_modus in ("visualisierung", "erkennung"):
        pca = PCA().fit(X_scaled)
        n_pca = _bestimme_pca_komponenten(pca, pca_komponenten, pca_varianz_schwelle)
        X_pca_full = pca.transform(X_scaled)

    # 3) Welche Daten gehen in DBSCAN?
    if pca_modus == "erkennung":
        X_for_dbscan = X_pca_full[:, :n_pca]
    else:
        X_for_dbscan = X_scaled

    # 4) Parameter-Defaults auf Basis der tatsächlichen DBSCAN-Dimension
    if min_samples is None:
        min_samples = max(4, 2 * X_for_dbscan.shape[1])

    eps_auto, k_distances, knee_idx = bestimme_eps(X_for_dbscan, min_samples)
    if eps is None:
        eps = eps_auto

    # 5) DBSCAN
    labels = DBSCAN(eps=eps, min_samples=min_samples).fit_predict(X_for_dbscan)
    anomalie_maske = labels == -1

    return {
        "labels": labels,
        "anomalie": anomalie_maske,
        "eps": eps,
        "min_samples": min_samples,
        "n_cluster": len(set(labels)) - (1 if -1 in labels else 0),
        "n_anomalien": int(anomalie_maske.sum()),
        "k_distances": k_distances,
        "knee_idx": knee_idx,
        "X_scaled": X_scaled,
        "scaler": scaler,
        "feature_namen": feature_namen_list,
        "pca_modus": pca_modus,
        "pca": pca,
        "X_pca": X_pca_full,
        "n_pca_komponenten": n_pca,
    }


# ---------------------------------------------------------------------------
# Visualisierung
# ---------------------------------------------------------------------------

FARBE_NORMAL = "#375A82"
FARBE_ANOMALIE = "#C24545"


def plot_k_distance(ergebnis: dict, speicherpfad: str | None = None) -> plt.Figure:
    """k-Distance-Plot mit markiertem Knick und eps-Linie."""
    sns.set_theme(style="whitegrid", context="notebook")
    fig, ax = plt.subplots(figsize=(8, 5))

    k = ergebnis["min_samples"]
    kd = ergebnis["k_distances"]
    knee = ergebnis["knee_idx"]
    eps = ergebnis["eps"]

    sns.lineplot(x=np.arange(len(kd)), y=kd, ax=ax, color=FARBE_NORMAL, linewidth=1.6)
    ax.axvline(knee, color=FARBE_ANOMALIE, linestyle="--", linewidth=1,
               label=f"Knick bei Index {knee}")
    ax.axhline(eps, color=FARBE_ANOMALIE, linestyle=":", linewidth=1,
               label=f"eps = {eps:.4f}")
    ax.set_xlabel("Punkte (nach k-Distanz sortiert)")
    ax.set_ylabel(f"Distanz zum {k}. Nachbarn")
    ax.set_title(f"k-Distance-Plot (k = {k}) zur eps-Bestimmung")
    ax.legend(loc="upper left")

    plt.tight_layout()
    if speicherpfad:
        fig.savefig(speicherpfad, dpi=140, bbox_inches="tight")
    return fig


def plot_scree(ergebnis: dict, speicherpfad: str | None = None) -> plt.Figure:
    """Scree-Plot: Einzel- und kumulative Varianz der PCA-Komponenten."""
    if ergebnis.get("pca") is None:
        raise ValueError("Scree-Plot nur verfügbar, wenn pca_modus != 'keine'.")

    sns.set_theme(style="whitegrid", context="notebook")
    var_ratio = ergebnis["pca"].explained_variance_ratio_
    cum_var = np.cumsum(var_ratio)
    n_sel = ergebnis["n_pca_komponenten"]

    fig, ax1 = plt.subplots(figsize=(10, 5))
    x = np.arange(1, len(var_ratio) + 1)

    # Einzelvarianz als Balken
    ax1.bar(x, var_ratio, color=FARBE_NORMAL, alpha=0.75, label="Einzelvarianz")
    ax1.set_xlabel("Hauptkomponente")
    ax1.set_ylabel("Erklärte Varianz (Anteil)", color=FARBE_NORMAL)
    ax1.set_xticks(x)
    ax1.tick_params(axis="y", labelcolor=FARBE_NORMAL)

    # Kumulative Varianz auf zweiter Achse
    ax2 = ax1.twinx()
    ax2.plot(x, cum_var, color=FARBE_ANOMALIE, marker="o",
             linewidth=1.8, label="Kumulativ")
    ax2.set_ylabel("Kumulative Varianz", color=FARBE_ANOMALIE)
    ax2.set_ylim(0, 1.05)
    ax2.tick_params(axis="y", labelcolor=FARBE_ANOMALIE)
    ax2.grid(False)

    # Schwellwert-Linien 90 % und 95 %
    for threshold in (0.90, 0.95):
        ax2.axhline(threshold, color="gray", linestyle=":", linewidth=0.8, alpha=0.6)
        ax2.text(len(var_ratio) + 0.1, threshold, f" {int(threshold * 100)} %",
                 color="gray", fontsize=9, va="center")

    # Markierung der ausgewählten Komponentenzahl
    ax1.axvline(n_sel + 0.5, color="black", linestyle="--", linewidth=1,
                alpha=0.5, label=f"Ausgewählt: {n_sel} Komp.")

    # Legende
    l1, lab1 = ax1.get_legend_handles_labels()
    l2, lab2 = ax2.get_legend_handles_labels()
    ax1.legend(l1 + l2, lab1 + lab2, loc="center right")

    ax1.set_title(
        f"Scree-Plot: {n_sel} von {len(var_ratio)} Komponenten "
        f"({cum_var[n_sel - 1] * 100:.1f} % Varianz)"
    )

    plt.tight_layout()
    if speicherpfad:
        fig.savefig(speicherpfad, dpi=140, bbox_inches="tight")
    return fig


def plot_pca_loadings(ergebnis: dict, speicherpfad: str | None = None) -> plt.Figure:
    """
    Heatmap der PCA-Loadings: wie stark trägt jedes Original-Merkmal zu jeder
    ausgewählten Hauptkomponente bei? Positive und negative Werte werden über
    ein divergentes Farbschema unterschieden (rot = positiv, blau = negativ).
    Nur bei aktivem PCA-Modus verfügbar.
    """
    if ergebnis.get("pca") is None:
        raise ValueError("Loadings-Plot nur verfügbar, wenn pca_modus != 'keine'.")

    sns.set_theme(style="white", context="notebook")

    pca = ergebnis["pca"]
    feature_namen = ergebnis["feature_namen"]
    n_sel = ergebnis["n_pca_komponenten"]
    var_ratio = pca.explained_variance_ratio_

    # Loadings-Matrix: Zeilen = Original-Merkmale, Spalten = Hauptkomponenten
    loadings = pca.components_[:n_sel].T
    pc_labels = [f"PC{i + 1}\n({var_ratio[i] * 100:.1f} %)" for i in range(n_sel)]

    fig, ax = plt.subplots(
        figsize=(max(6.0, 1.2 * n_sel + 3), max(4.0, 0.45 * len(feature_namen) + 1.5))
    )
    sns.heatmap(
        loadings,
        xticklabels=pc_labels,
        yticklabels=feature_namen,
        annot=True, fmt=".2f",
        cmap="RdBu_r", center=0, vmin=-1, vmax=1,
        linewidths=0.4, linecolor="white",
        cbar_kws={"label": "Loading"},
        ax=ax,
    )
    ax.set_title(
        "PCA-Loadings: Beitrag der Original-Merkmale zu den Hauptkomponenten"
    )
    ax.set_xlabel("")
    ax.set_ylabel("")
    plt.setp(ax.get_yticklabels(), rotation=0)
    plt.setp(ax.get_xticklabels(), rotation=0)

    plt.tight_layout()
    if speicherpfad:
        fig.savefig(speicherpfad, dpi=140, bbox_inches="tight")
    return fig


def _wrap_an_trennern(text: str, max_zeichen: int = 16) -> str:
    """
    Bricht einen Text um: bevorzugt an natürlichen Trennern (_, -, Leerzeichen),
    hart wenn ein einzelnes Token länger als max_zeichen ist.
    """
    if not text or len(text) <= max_zeichen:
        return text
    import re
    tokens = [t for t in re.split(r"([_\- ])", text) if t]

    # Zu lange Tokens hart aufteilen
    def _split_hart(tok: str) -> list[str]:
        if len(tok) <= max_zeichen or tok in "_- ":
            return [tok]
        parts = []
        while len(tok) > max_zeichen:
            parts.append(tok[:max_zeichen])
            tok = tok[max_zeichen:]
        if tok:
            parts.append(tok)
        return parts

    tokens_klein: list[str] = []
    for t in tokens:
        tokens_klein.extend(_split_hart(t))

    lines: list[str] = []
    aktuell = ""
    for tok in tokens_klein:
        if len(aktuell) + len(tok) > max_zeichen and aktuell.strip():
            lines.append(aktuell.rstrip("_- "))
            aktuell = tok.lstrip("_- ")
        else:
            aktuell += tok
    if aktuell.strip():
        lines.append(aktuell.rstrip("_- "))
    return "\n".join(lines)


def _labels_lesbar_machen(g: sns.axisgrid.PairGrid, feature_namen: list[str],
                          max_zeichen: int = 16) -> None:
    """
    Macht Achsenbeschriftungen einer PairGrid auch bei langen Namen lesbar.
    Tick-Labels (kurze Zahlen) bleiben horizontal — nur die Achsen-Titel
    werden rotiert und gewrappt.
    """
    if not feature_namen:
        return
    max_len = max(len(str(n).replace("\n", " ")) for n in feature_namen)
    braucht_wrap = max_len > max_zeichen
    braucht_rotation = max_len > 8
    winkel = 45 if max_len > 12 else 25

    def _neu(text: str) -> str:
        return _wrap_an_trennern(text, max_zeichen) if braucht_wrap else text

    # X-Achsen-Labels (unten): wrap + rotate
    for ax in g.axes[-1, :]:
        if ax is None:
            continue
        xl = ax.get_xlabel()
        if xl:
            if braucht_rotation:
                ax.set_xlabel(_neu(xl), rotation=winkel, ha="right",
                              rotation_mode="anchor")
            else:
                ax.set_xlabel(_neu(xl))

    # Y-Achsen-Labels (links): nur wrap, keine Rotation
    for ax in g.axes[:, 0]:
        if ax is None:
            continue
        yl = ax.get_ylabel()
        if yl:
            ax.set_ylabel(_neu(yl))

    # Ränder anpassen. Anzahl Zeilen im gewrappten Label bestimmt Bedarf.
    max_zeilen = 1
    if braucht_wrap:
        for n in feature_namen:
            max_zeilen = max(max_zeilen, len(_wrap_an_trennern(str(n), max_zeichen).split("\n")))
    bottom_pad = min(0.30, 0.10 + 0.045 * max_zeilen + (0.06 if braucht_rotation else 0))
    left_pad = min(0.22, 0.09 + 0.025 * max_zeilen)
    g.fig.subplots_adjust(bottom=bottom_pad, left=left_pad)


def plot_scatter_matrix(
    daten: np.ndarray,
    ergebnis: dict,
    feature_namen: np.ndarray | list[str] | None = None,
    verwende_pca: bool | None = None,
    speicherpfad: str | None = None,
) -> sns.axisgrid.PairGrid:
    """
    Scatter-Plot-Matrix mit farblich hervorgehobenen Anomalien.

    Args:
        verwende_pca: True → Matrix der PCA-Komponenten. False → Originaldaten.
                      None (Default) → automatisch: PCA wenn verfügbar, sonst Original.
    """
    if verwende_pca is None:
        verwende_pca = ergebnis.get("pca") is not None

    if verwende_pca:
        if ergebnis.get("X_pca") is None:
            raise ValueError("verwende_pca=True, aber keine PCA im Ergebnis vorhanden.")
        n_comp = ergebnis["n_pca_komponenten"]
        plot_daten = ergebnis["X_pca"][:, :n_comp]
        plot_namen = [f"PC{i + 1}" for i in range(n_comp)]
        varianz = ergebnis["pca"].explained_variance_ratio_[:n_comp]
        # Achsenbeschriftungen mit erklärter Varianz anreichern
        plot_namen = [f"{name}\n({v * 100:.1f} %)" for name, v in zip(plot_namen, varianz)]
        titel_zusatz = f"(PCA-Projektion, {n_comp} Komp.)"
    else:
        plot_daten = daten
        if feature_namen is None:
            # Namen aus Ergebnis-Dict übernehmen, sonst Defaults
            feature_namen = ergebnis.get(
                "feature_namen", [f"x{i + 1}" for i in range(daten.shape[1])]
            )
        plot_namen = [str(x) for x in feature_namen]
        titel_zusatz = "(Originaldaten)"

    df = pd.DataFrame(plot_daten, columns=plot_namen)
    df["Typ"] = np.where(ergebnis["anomalie"], "Anomalie", "Normal")
    df["Typ"] = pd.Categorical(df["Typ"], categories=["Normal", "Anomalie"], ordered=True)
    df = df.sort_values("Typ")

    # Höhe der Einzel-Subplots adaptiv: bei vielen Merkmalen kleiner,
    # damit die Gesamtmatrix nicht ins Riesenformat wächst.
    n_shown = len(plot_namen)
    height = 2.2 if n_shown <= 5 else max(1.3, 11.0 / n_shown)

    sns.set_theme(style="whitegrid", context="notebook")
    g = sns.pairplot(
        df, vars=plot_namen, hue="Typ",
        palette={"Normal": FARBE_NORMAL, "Anomalie": FARBE_ANOMALIE},
        markers=["o", "X"],
        plot_kws={"alpha": 0.6, "s": 30, "edgecolor": "none"},
        diag_kind="hist",
        diag_kws={"alpha": 0.6, "bins": 25, "edgecolor": "white", "linewidth": 0.3},
        height=height,
        corner=False,
    )

    # Lange Labels lesbar machen: wrappen, x-Achse rotieren, mehr Rand
    _labels_lesbar_machen(g, plot_namen)
    g.fig.suptitle(
        f"DBSCAN: {ergebnis['n_cluster']} Cluster + {ergebnis['n_anomalien']} Anomalien "
        f"{titel_zusatz}",
        y=1.02, fontsize=12,
    )
    if speicherpfad:
        g.fig.savefig(speicherpfad, dpi=140, bbox_inches="tight")
    return g


def visualisiere(
    daten: np.ndarray,
    ergebnis: dict,
    feature_namen: np.ndarray | list[str] | None = None,
    speicherpfad_k: str | None = None,
    speicherpfad_scree: str | None = None,
    speicherpfad_loadings: str | None = None,
    speicherpfad_matrix_original: str | None = None,
    speicherpfad_matrix_pca: str | None = None,
) -> dict:
    """
    Erzeugt alle relevanten Plots.

    Bei jedem Aufruf:
      - k-Distance-Plot
      - Scatter-Matrix der Original-Rohdaten (mit feature_namen aus Ergebnis)

    Zusätzlich bei aktivem PCA-Modus:
      - Scree-Plot der erklärten Varianz
      - Loadings-Heatmap (Beitrag der Original-Merkmale zu den PCs)
      - Scatter-Matrix der PCA-Komponenten
    """
    figs = {}
    figs["k_distance"] = plot_k_distance(ergebnis, speicherpfad_k)

    # Immer: Scatter-Matrix der Originaldaten
    figs["matrix_original"] = plot_scatter_matrix(
        daten, ergebnis, feature_namen,
        verwende_pca=False,
        speicherpfad=speicherpfad_matrix_original,
    )

    if ergebnis.get("pca") is not None:
        figs["scree"] = plot_scree(ergebnis, speicherpfad_scree)
        figs["loadings"] = plot_pca_loadings(ergebnis, speicherpfad_loadings)
        # Zusätzlich: Scatter-Matrix der PCA-Komponenten
        figs["matrix_pca"] = plot_scatter_matrix(
            daten, ergebnis, feature_namen=None,
            verwende_pca=True,
            speicherpfad=speicherpfad_matrix_pca,
        )

    return figs


In [ ]:
import pickle
import numpy

pickle_file = 'data/df_piv_02.pkl'
with open(pickle_file, "rb") as f:
    df = pickle.load(f)
df = df.reset_index()
df = df.set_index("timestamp")
df = df[df.MPG08_Domaene == 'P0C00Y00_PERM01_2201-ALLEGRO11A_2']
df_num = df.select_dtypes(include=['float64'])

feature_namen = df_num.columns 
daten = df_num.to_numpy()
n_features = 20

In [ ]:
if __name__ == "__main__":

    print(f"Rohdaten: shape = {daten.shape}   ({n_features} Merkmale)")
    print(f"Feature-Namen: {feature_namen.tolist()}\n")

    # --- DBSCAN mit PCA nur für die Visualisierung ---
    # feature_namen einmal übergeben — landen im Ergebnis-Dict und werden
    # von den Plot-Funktionen automatisch übernommen.
    ergebnis = dbscan_anomalien(
        daten,
        feature_namen=feature_namen,
        pca_modus="visualisierung",
        pca_varianz_schwelle=0.90,
    )

    print(f"pca_modus:        {ergebnis['pca_modus']}")
    print(f"PCA-Komponenten:  {ergebnis['n_pca_komponenten']} "
          f"(90 % Varianz-Schwelle)")
    print(f"DBSCAN-Parameter: eps = {ergebnis['eps']:.4f}, "
          f"min_samples = {ergebnis['min_samples']}")
    print(f"Cluster:          {ergebnis['n_cluster']}")
    print(f"Anomalien:        {ergebnis['n_anomalien']} von {len(daten)} "
          f"({100 * ergebnis['n_anomalien'] / len(daten):.1f} %)\n")

    # --- Visualisierung ---
    # feature_namen muss hier nicht mehr übergeben werden — sie kommen
    # automatisch aus ergebnis["feature_namen"].
    visualisiere(
        daten, ergebnis,
        speicherpfad_k="dbscan_k_distance.png",
        speicherpfad_scree="dbscan_scree.png",
        speicherpfad_loadings="dbscan_loadings.png",
        speicherpfad_matrix_original="dbscan_matrix_original.png",
        speicherpfad_matrix_pca="dbscan_matrix_pca.png",
    )
    plt.show()

    # --- Alternative: PCA auch für Erkennung ---
    # ergebnis2 = dbscan_anomalien(daten, feature_namen=feature_namen,
    #                              pca_modus="erkennung",
    #                              pca_varianz_schwelle=0.90)
    # visualisiere(daten, ergebnis2)